# HWANGE — Colab driver

This notebook runs the pipeline; it does not contain it. All logic lives in `src/`.
Set the runtime to a **GPU** before running (Runtime > Change runtime type > T4).


## 1. Clone and install

In [ ]:
!git clone https://github.com/semereherruy/TRI-AI-hwange-proj.git
%cd TRI-AI-hwange-proj
!pip install -q -r requirements.txt
!python check_env.py

### Fail-fast helper\n\nStops the notebook at the first failing step instead of cascading.

In [ ]:
# Shell cells with `!` do not raise when a command fails, and subprocess output
# goes to the kernel rather than the cell. This helper captures both streams,
# prints them, and stops the notebook at the actual failure.
import subprocess, sys

def run(command: str) -> None:
    print(f"$ {command}", flush=True)
    result = subprocess.run(
        command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    print(result.stdout, flush=True)
    if result.returncode != 0:
        raise SystemExit(f"STOPPED (exit {result.returncode}): {command}")
    print("ok\n", flush=True)

## 2. Credentials

Gemma and AfriHate are gated. Add your token to the Colab **Secrets** panel (key icon)
as `HF_TOKEN` — never paste it into a cell.

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("token loaded:", bool(os.environ.get("HF_TOKEN")))

## 3. Fetch the third-party datasets

The vendored repos are git-ignored, so HateXplain is re-fetched here. AfriHate and
ToxiGen come from the Hub via the loaders.

In [ ]:
run("mkdir -p data/raw")
run("git clone -q --depth 1 https://github.com/hate-alert/HateXplain.git data/raw/HateXplain-master || true")
run("ls data/raw/HateXplain-master/Data")

## 4. Data pipeline (Phases 2-7)

Each step writes a report. Edit `configs/data.yaml` to change a labelling policy;
every artefact records the policy that produced it.

In [ ]:
run("python -m src.data.inspection")      # Phase 2: inspection reports
run("python -m src.data.canonical")       # Phase 4: canonical dataset
run("python -m src.data.quality")         # Phase 5: quality + leakage report
run("python -m src.data.splits")          # Phase 6: group-level splits
run("python -m src.analysis.baselines")   # Phase 7: TF-IDF baselines

## 5. Gemma hidden-state extraction (Phase 8)

Smoke-test the path on a few rows first, then run the full extraction. The model is
frozen and used in inference mode only.

In [ ]:
# smoke-test the path on a few rows first, then the full extraction
run("python -m src.probing.extraction --splits train test --limit 32 --output /tmp/smoke")
run("python -m src.probing.extraction --splits train val test probe")

## 6. Layer-wise probes and controls (Phases 9-10)

In [ ]:
run("python -m src.probing.probes --embeddings data/embeddings")

## 7. Layer curve

The primary Phase 9 output: performance as a function of depth, read against the
Phase 7 TF-IDF baseline and the Phase 10 control floors.

In [ ]:
run("python -m src.probing.probes --embeddings data/embeddings")